In [1]:
/**
 * @file grafos_benchmark.cpp
 * @brief Implementación y comparación empírica (Benchmark) de representaciones de Grafos.
 * @standard C++17
 *
 * Este programa demuestra la diferencia en la vida real entre usar una Matriz 
 * de Adyacencia O(V^2) y una Lista de Adyacencia O(V+E). 
 * 
 */

#include <iostream>
#include <vector>
#include <cstdint>     // Para usar uint32_t (enteros sin signo de tamaño fijo)
#include <stdexcept>   // Para lanzar excepciones como std::out_of_range
#include <algorithm>   // Para algoritmos de la STL como std::find
#include <chrono>      // Para medir el tiempo de ejecución (benchmarking)
#include <memory>      // Para punteros inteligentes
#include <iomanip>     // Para formatear la salida en consola

// ============================================================================
// 1. CONTRATO DE INTERFAZ (TIPO DE DATO ABSTRACTO)
// ============================================================================

class IGraph {
public:
    // Destructor virtual: ¡CRÍTICO! Evita fugas de memoria al usar polimorfismo.
    virtual ~IGraph() = default;

    // Método para conectar dos vértices (u, v)
    virtual void addEdge(uint32_t u, uint32_t v) = 0;

    // 'const' al final indica que esta función NO modificará los datos del grafo.
    virtual bool hasEdge(uint32_t u, uint32_t v) const = 0;

    // Retorna todos los vecinos de un vértice u.
    virtual std::vector<uint32_t> getNeighbors(uint32_t u) const = 0;

    // 'noexcept' promete al compilador que esta función JAMÁS lanzará un error.
    virtual uint32_t getVertexCount() const noexcept = 0;
};

// ============================================================================
// 2. MATRIZ DE ADYACENCIA (Complejidad Espacial: O(V^2))
// ============================================================================

class AdjacencyMatrix : public IGraph {
private:
    uint32_t numVertices;
    
    // Matriz 2D. 
    // DATO: En C++, std::vector<bool> implementa "Bit-Packing",
    // guardando 8 booleanos en 1 solo byte de RAM para ahorrar memoria.
    std::vector<std::vector<bool>> matrix;

    inline void validateVertex(uint32_t v) const {
        if (v >= numVertices) {
            throw std::out_of_range("Identificador de vértice fuera de los límites.");
        }
    }

public:
    explicit AdjacencyMatrix(uint32_t vertices) : numVertices(vertices) {
        matrix.resize(numVertices, std::vector<bool>(numVertices, false));
    }

    void addEdge(uint32_t u, uint32_t v) override {
        validateVertex(u);
        validateVertex(v);
        // Grafo no dirigido = matriz simétrica. Tiempo: O(1)
        matrix[u][v] = true;
        matrix[v][u] = true; 
    }

    bool hasEdge(uint32_t u, uint32_t v) const override {
        validateVertex(u);
        validateVertex(v);
        // Tiempo: O(1) - Acceso directo, la mayor ventaja de la Matriz.
        return matrix[u][v];
    }

    std::vector<uint32_t> getNeighbors(uint32_t u) const override {
        validateVertex(u);
        std::vector<uint32_t> neighbors;
        neighbors.reserve(numVertices / 2); 
        
        // ¡CUELLO DE BOTELLA! (Tiempo: O(V))
        // Revisa toda la fila, incluso los ceros. Fatal en grafos dispersos.
        for (uint32_t v = 0; v < numVertices; ++v) {
            if (matrix[u][v]) {
                neighbors.push_back(v);
            }
        }
        return neighbors;
    }

    uint32_t getVertexCount() const noexcept override {
        return numVertices;
    }
};

// ============================================================================
// 3. LISTA DE ADYACENCIA (Complejidad Espacial: O(V + E))
// ============================================================================

class AdjacencyList : public IGraph {
private:
    uint32_t numVertices;
    
    // Usamos arreglo de arreglos. Garantiza "Memoria Contigua", 
    // permitiendo que la CPU cargue vecinos de golpe en caché L1 (Cache Hits).
    std::vector<std::vector<uint32_t>> adj;

    inline void validateVertex(uint32_t v) const {
        if (v >= numVertices) {
            throw std::out_of_range("Identificador de vértice fuera de los límites.");
        }
    }

public:
    explicit AdjacencyList(uint32_t vertices) : numVertices(vertices) {
        adj.resize(numVertices); 
    }

    void addEdge(uint32_t u, uint32_t v) override {
        validateVertex(u);
        validateVertex(v);
        // O(1) amortizado
        adj[u].push_back(v);
        adj[v].push_back(u);
    }

    bool hasEdge(uint32_t u, uint32_t v) const override {
        validateVertex(u);
        const auto& neighbors = adj[u];
        // Búsqueda lineal: O(deg(u)). La Lista pierde frente a la Matriz aquí.
        return std::find(neighbors.begin(), neighbors.end(), v) != neighbors.end();
    }

    std::vector<uint32_t> getNeighbors(uint32_t u) const override {
        validateVertex(u);
        // Tiempo: O(deg(u)). Optimizado por Copy Elision (NRVO).
        return adj[u];
    }

    uint32_t getVertexCount() const noexcept override {
        return numVertices;
    }
};

// ============================================================================
// 4. MÓDULO DE BENCHMARKING (ADAPTADO PARA NOTEBOOKS)
// ============================================================================

class GraphBenchmark {
private:
    static double measureNeighborsTraversal(const IGraph& graph) {
        auto start = std::chrono::high_resolution_clock::now();
        uint32_t V = graph.getVertexCount();
        
        // 'volatile' obliga al compilador/intérprete a no saltarse este bucle.
        volatile size_t dummyCount = 0; 
        
        for (uint32_t i = 0; i < V; ++i) {
            auto neighbors = graph.getNeighbors(i);
            dummyCount += neighbors.size();
        }

        auto end = std::chrono::high_resolution_clock::now();
        std::chrono::duration<double, std::milli> diff = end - start;
        return diff.count();
    }

public:
    static void runSparseGraphTest() {
        std::cout << "\n[PRUEBA 1] GRAFO DISPERSO (Ej. Red Social / Molecula)\n";
        constexpr uint32_t V = 2000; 
        std::cout << "Topologia: V = " << V << ", Aristas por vertice = 3 (Total ~6k aristas)\n";

        AdjacencyMatrix mat(V);
        AdjacencyList lst(V);

        for (uint32_t i = 0; i < V; ++i) {
            uint32_t target1 = (i + 1) % V;
            uint32_t target2 = (i + 2) % V;
            uint32_t target3 = (i + 3) % V;
            mat.addEdge(i, target1); mat.addEdge(i, target2); mat.addEdge(i, target3);
            lst.addEdge(i, target1); lst.addEdge(i, target2); lst.addEdge(i, target3);
        }

        std::cout << "Midiendo operacion getNeighbors() en todo el grafo...\n";
        double timeMat = measureNeighborsTraversal(mat);
        double timeLst = measureNeighborsTraversal(lst);

        std::cout << " -> Tiempo MATRIZ O(V^2)  : " << std::fixed << std::setprecision(4) << timeMat << " ms\n";
        std::cout << " -> Tiempo LISTA O(V+E)   : " << std::fixed << std::setprecision(4) << timeLst << " ms\n";
        std::cout << " -> [ANALISIS] La Matriz sufrio al tener que recorrer miles de ceros vacios.\n";
    }

    static void runDenseGraphTest() {
        std::cout << "\n[PRUEBA 2] GRAFO DENSO (Ej. Red de Enrutadores Local)\n";
        constexpr uint32_t V = 200;
        std::cout << "Topologia: V = " << V << ", Grafo Completamente Conectado (Total ~20k aristas)\n";

        AdjacencyMatrix mat(V);
        AdjacencyList lst(V);

        for (uint32_t i = 0; i < V; ++i) {
            for (uint32_t j = i + 1; j < V; ++j) {
                mat.addEdge(i, j);
                lst.addEdge(i, j);
            }
        }

        std::cout << "Midiendo operacion hasEdge() (100 mil consultas)...\n";
        
        auto startMat = std::chrono::high_resolution_clock::now();
        volatile size_t hitsMat = 0;
        for(uint32_t k = 0; k < 100000; ++k) {
            if(mat.hasEdge(k % V, (k + 100) % V)) hitsMat++;
        }
        auto endMat = std::chrono::high_resolution_clock::now();

        auto startLst = std::chrono::high_resolution_clock::now();
        volatile size_t hitsLst = 0;
        for(uint32_t k = 0; k < 100000; ++k) {
            if(lst.hasEdge(k % V, (k + 100) % V)) hitsLst++;
        }
        auto endLst = std::chrono::high_resolution_clock::now();

        std::chrono::duration<double, std::milli> diffMat = endMat - startMat;
        std::chrono::duration<double, std::milli> diffLst = endLst - startLst;

        std::cout << " -> Tiempo MATRIZ O(1)    : " << std::fixed << std::setprecision(4) << diffMat.count() << " ms\n";
        std::cout << " -> Tiempo LISTA O(V)     : " << std::fixed << std::setprecision(4) << diffLst.count() << " ms\n";
        std::cout << " -> La Lista sufre en grafos densos por busquedas secuenciales largas.\n\n";
    }
};

// ============================================================================
// 5. EJECUCIÓN DIRECTA PARA JUPYTER NOTEBOOK (SIN MAIN)
// ============================================================================

void test() {
    std::cout << "========================================================\n";
    std::cout << "   BENCHMARK EMPIRICO: MATRIZ VS LISTA DE ADYACENCIA    \n";
    std::cout << "========================================================\n";
    
    try {
        GraphBenchmark::runSparseGraphTest();
        GraphBenchmark::runDenseGraphTest();
    } catch (const std::exception& e) {
        std::cerr << "Excepcion critica: " << e.what() << '\n';
    }

    std::cout << "====================== FIN DEL TEST ====================\n";
}

// Llamada en el ámbito global. Un kernel C++ evaluará esto inmediatamente.
test();

   BENCHMARK EMPIRICO: MATRIZ VS LISTA DE ADYACENCIA    

[PRUEBA 1] GRAFO DISPERSO (Ej. Red Social / Molecula)
Topologia: V = 2000, Aristas por vertice = 3 (Total ~6k aristas)
Midiendo operacion getNeighbors() en todo el grafo...
 -> Tiempo MATRIZ O(V^2)  : 59.4138 ms
 -> Tiempo LISTA O(V+E)   : 0.3198 ms
 -> [ANALISIS] La Matriz sufrio al tener que recorrer miles de ceros vacios.

[PRUEBA 2] GRAFO DENSO (Ej. Red de Enrutadores Local)
Topologia: V = 200, Grafo Completamente Conectado (Total ~20k aristas)
Midiendo operacion hasEdge() (100 mil consultas)...
 -> Tiempo MATRIZ O(1)    : 2.3562 ms
 -> Tiempo LISTA O(V)     : 63.2214 ms
 -> [ANALISIS] La Lista sufre en grafos densos por busquedas secuenciales largas.

====================== FIN DEL TEST ====================
